In [1]:
from pathlib import Path
import sys

sys.path.append(str(Path.cwd().parent))

In [2]:
from torch.utils.data import random_split, DataLoader
from src.configs import SEED, BATCH_SIZE
from src.dataset import ImageDataset
from src.loss_function import Loss
from src.model import Model
from pathlib import Path
import torch

In [3]:
from torchvision.transforms import v2
import torch

# The mean and standard deviations across each channel for the normalized pixels
# of every single image in the "trainval" dataset
MEANS = (0.4485, 0.4249, 0.3922)
STDS = (0.2682, 0.2655, 0.2782)

trainval_transforms = v2.Compose([
    v2.Normalize(mean=MEANS, std=STDS)
])

In [4]:
annot_file_trainval = Path("../data/preprocessed/trainval/annotations.csv")
img_dir_trainval = Path("../data/preprocessed/trainval/Images")

trainval_dataset = ImageDataset(annot_file_trainval, img_dir_trainval,
                                transform=trainval_transforms)

generator_ = torch.Generator().manual_seed(SEED)
train_dataset, val_dataset = random_split(trainval_dataset, [0.05, 0.95]
                                          ,generator=generator_)

len(train_dataset)

251

In [5]:
train_dl = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [7]:
num_epochs = 8

In [7]:
model = Model().to(device)
loss_fn = Loss().to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [12]:
for epoch in range(20):
    model.train()
    epoch_loss = 0.0
    loss_1 = 0.0
    loss_2 = 0.0
    loss_3 = 0.0
    loss_4 = 0.0
    loss_5 = 0.0

    for X_batch, y_batch in train_dl:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        preds = model(X_batch)
        losses = loss_fn(preds, y_batch)
        loss = losses["total_loss"]

        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        loss_1 += losses["loss_1"]
        loss_2 += losses["loss_2"]
        loss_3 += losses["loss_3"]
        loss_4 += losses["loss_4"]
        loss_5 += losses["loss_5"]

    print(epoch, epoch_loss / len(train_dl))
    print(epoch, loss_1 / len(train_dl))
    print(epoch, loss_2 / len(train_dl))
    print(epoch, loss_3 / len(train_dl))
    print(epoch, loss_4 / len(train_dl))
    print(epoch, loss_5 / len(train_dl))

0 3.3882156014442444
0 tensor(0.1962, device='cuda:0', grad_fn=<DivBackward0>)
0 tensor(0.2412, device='cuda:0', grad_fn=<DivBackward0>)
0 tensor(0.8287, device='cuda:0', grad_fn=<DivBackward0>)
0 tensor(0.7728, device='cuda:0', grad_fn=<DivBackward0>)
0 tensor(1.3492, device='cuda:0', grad_fn=<DivBackward0>)
1 3.354815572500229
1 tensor(0.1942, device='cuda:0', grad_fn=<DivBackward0>)
1 tensor(0.2306, device='cuda:0', grad_fn=<DivBackward0>)
1 tensor(0.7994, device='cuda:0', grad_fn=<DivBackward0>)
1 tensor(0.7465, device='cuda:0', grad_fn=<DivBackward0>)
1 tensor(1.3841, device='cuda:0', grad_fn=<DivBackward0>)
2 3.2850186228752136
2 tensor(0.1873, device='cuda:0', grad_fn=<DivBackward0>)
2 tensor(0.2114, device='cuda:0', grad_fn=<DivBackward0>)
2 tensor(0.7616, device='cuda:0', grad_fn=<DivBackward0>)
2 tensor(0.7108, device='cuda:0', grad_fn=<DivBackward0>)
2 tensor(1.4140, device='cuda:0', grad_fn=<DivBackward0>)
3 3.230598747730255
3 tensor(0.1877, device='cuda:0', grad_fn=<DivBa

In [14]:
checkpoint = {
    "epoch": epoch+1,
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
}

checkpoint_dir = Path("../outputs/checkpoints")
torch.save(checkpoint, checkpoint_dir / f"checkpoint_epoch_10.pth")

In [8]:
checkpoint = torch.load("../outputs/checkpoints/checkpoint_epoch_10.pth")
model.load_state_dict(checkpoint["model_state_dict"])
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

In [9]:
from src.inference_functions import compute_eval_stats

train_mAP = compute_eval_stats(model, train_dl, device)
train_mAP

0.8308367385483157

In [ ]:
list(range(15, 75+1, 5))

In [ ]:
list(range(0+1, 15+1))

In [ ]:
import time

start_time = time.perf_counter()

time.sleep(5)

end_time = time.perf_counter()

end_time - start_time